<a href="https://colab.research.google.com/github/kevinhun1/Air-Quality-Monitoring-System/blob/PM2.5_ML_classifier/Air_Quality_RF_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

excel_file_path = '/content/drive/MyDrive/Air Quality RF Classifier Data set/Dataset_air_quality_2025.xlsx'
df = pd.read_excel(excel_file_path)

# Assuming the semicolon-separated data is in the first column
df = df.iloc[:, 0].str.split(';', expand=True)

# Assign column names based on the expected structure
df.columns = ['sensor_id', 'sensor_type', 'location', 'lat', 'lon', 'timestamp', 'value_type', 'value']

display(df)

,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value
0,79,SDS011,30,-1.290,36.777,2019-04-01T00:00:17.141770+00:00,P1,14.80
1,79,SDS011,30,-1.290,36.777,2019-04-01T00:00:17.141770+00:00,P2,8.23
2,132,SDS011,60,-1.364,36.915,2019-04-01T00:00:18.304763+00:00,P2,10.65
3,132,SDS011,60,-1.364,36.915,2019-04-01T00:00:18.304763+00:00,P1,19.83
4,80,DHT22,30,-1.290,36.777,2019-04-01T00:00:18.769957+00:00,temperature,21.90
...,...,...,...,...,...,...,...,...
1067,58,DHT22,29,-1.300,36.785,2019-04-01T01:22:14.370152+00:00,temperature,21.50
1068,69,SDS011,61,-1.365,36.914,2019-04-01T01:22:27.504366+00:00,P2,6.683316192626951
1069,69,SDS011,61,-1.365,36.914,2019-04-01T01:22:27.504366+00:00,P1,17.2728628540039
1070,70,DHT22,61,-1.365,36.914,2019-04-01T01:22:27.549697+00:00,temperature,19.29931640625


In [ ]:
# Save the processed DataFrame to an Excel file
output_excel_path = '/content/df_classified.xlsx'
df.to_excel(output_excel_path, index=False)

print(f"Processed data saved to {output_excel_path}")

Processed data saved to /content/df_classified.xlsx


In [ ]:
# Check for missing values
print("Missing values before cleaning:")
display(df.isnull().sum())

# Drop rows with any missing values
df.dropna(inplace=True)

# Check for missing values after dropping
print("\nMissing values after dropping:")
display(df.isnull().sum())

# Convert 'value' column to numeric, coercing errors to NaN
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Drop rows where 'value' could not be converted to numeric (became NaN)
df.dropna(subset=['value'], inplace=True)

# Convert 'timestamp' to datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("\nDataFrame after cleaning:")
display(df.head())
display(df.info())

Missing values before cleaning:


,0
sensor_id,0
sensor_type,0
location,0
lat,0
lon,0
timestamp,0
value_type,0
value,0



Missing values after dropping:


,0
sensor_id,0
sensor_type,0
location,0
lat,0
lon,0
timestamp,0
value_type,0
value,0



DataFrame after cleaning:


,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value
0,79,SDS011,30,-1.290,36.777,2019-04-01 00:00:17.141770+00:00,P1,14.80
1,79,SDS011,30,-1.290,36.777,2019-04-01 00:00:17.141770+00:00,P2,8.23
2,132,SDS011,60,-1.364,36.915,2019-04-01 00:00:18.304763+00:00,P2,10.65
3,132,SDS011,60,-1.364,36.915,2019-04-01 00:00:18.304763+00:00,P1,19.83
4,80,DHT22,30,-1.290,36.777,2019-04-01 00:00:18.769957+00:00,temperature,21.90


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072 entries, 0 to 1071
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   sensor_id    1072 non-null   object             
 1   sensor_type  1072 non-null   object             
 2   location     1072 non-null   object             
 3   lat          1072 non-null   object             
 4   lon          1072 non-null   object             
 5   timestamp    1072 non-null   datetime64[ns, UTC]
 6   value_type   1072 non-null   object             
 7   value        1072 non-null   float64            
dtypes: datetime64[ns, UTC](1), float64(1), object(6)
memory usage: 67.1+ KB


None

In [ ]:
df_selected = df[['sensor_id', 'value', 'timestamp']]
display(df_selected.head())

,sensor_id,value,timestamp
0,79,14.80,2019-04-01 00:00:17.141770+00:00
1,79,8.23,2019-04-01 00:00:17.141770+00:00
2,132,10.65,2019-04-01 00:00:18.304763+00:00
3,132,19.83,2019-04-01 00:00:18.304763+00:00
4,80,21.90,2019-04-01 00:00:18.769957+00:00


In [ ]:
# Extract time-based features from the 'timestamp' column
df_selected['year'] = df_selected['timestamp'].dt.year
df_selected['month'] = df_selected['timestamp'].dt.month
df_selected['day'] = df_selected['timestamp'].dt.day
df_selected['hour'] = df_selected['timestamp'].dt.hour
df_selected['minute'] = df_selected['timestamp'].dt.minute
df_selected['dayofweek'] = df_selected['timestamp'].dt.dayofweek # Monday=0, Sunday=6
df_selected['weekofyear'] = df_selected['timestamp'].dt.isocalendar().week.astype(int)

display(df_selected.head())

/tmp/ipython-input-864015833.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['year'] = df_selected['timestamp'].dt.year
/tmp/ipython-input-864015833.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['month'] = df_selected['timestamp'].dt.month
/tmp/ipython-input-864015833.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.p

,sensor_id,value,timestamp,year,month,day,hour,minute,dayofweek,weekofyear
0,79,14.80,2019-04-01 00:00:17.141770+00:00,2019,4,1,0,0,0,14
1,79,8.23,2019-04-01 00:00:17.141770+00:00,2019,4,1,0,0,0,14
2,132,10.65,2019-04-01 00:00:18.304763+00:00,2019,4,1,0,0,0,14
3,132,19.83,2019-04-01 00:00:18.304763+00:00,2019,4,1,0,0,0,14
4,80,21.90,2019-04-01 00:00:18.769957+00:00,2019,4,1,0,0,0,14


In [ ]:
# Extract time-based features from the 'timestamp' column
df_selected = df_selected.copy() # Explicitly create a copy to avoid SettingWithCopyWarning
df_selected['year'] = df_selected['timestamp'].dt.year
df_selected['month'] = df_selected['timestamp'].dt.month
df_selected['day'] = df_selected['timestamp'].dt.day
df_selected['hour'] = df_selected['timestamp'].dt.hour
df_selected['minute'] = df_selected['timestamp'].dt.minute
df_selected['dayofweek'] = df_selected['timestamp'].dt.dayofweek # Monday=0, Sunday=6
df_selected['weekofyear'] = df_selected['timestamp'].dt.isocalendar().week.astype(int)

display(df_selected.head())

,sensor_id,value,timestamp,year,month,day,hour,minute,dayofweek,weekofyear
0,79,14.80,2019-04-01 00:00:17.141770+00:00,2019,4,1,0,0,0,14
1,79,8.23,2019-04-01 00:00:17.141770+00:00,2019,4,1,0,0,0,14
2,132,10.65,2019-04-01 00:00:18.304763+00:00,2019,4,1,0,0,0,14
3,132,19.83,2019-04-01 00:00:18.304763+00:00,2019,4,1,0,0,0,14
4,80,21.90,2019-04-01 00:00:18.769957+00:00,2019,4,1,0,0,0,14


In [ ]:
df_selected = df_selected.drop('sensor_id', axis=1)
display(df_selected.head())

,value,year,month,day,hour,minute,dayofweek,weekofyear
0,14.80,2019,4,1,0,0,0,14
1,8.23,2019,4,1,0,0,0,14
2,10.65,2019,4,1,0,0,0,14
3,19.83,2019,4,1,0,0,0,14
4,21.90,2019,4,1,0,0,0,14


In [ ]:
def who_label(value):
    if value <= 10:
        return 0   # Low
    elif value <= 15:
        return 1   # Moderate
    elif value <= 25:
        return 2   # Unhealthy
    elif value <= 35:
        return 3   # Very Unhealthy
    else:
        return 4   # Hazardous

df_selected["risk"] = df_selected["value"].apply(who_label)
df_selected.head()

,value,year,month,day,hour,minute,dayofweek,weekofyear,risk
0,14.80,2019,4,1,0,0,0,14,1
1,8.23,2019,4,1,0,0,0,14,0
2,10.65,2019,4,1,0,0,0,14,1
3,19.83,2019,4,1,0,0,0,14,2
4,21.90,2019,4,1,0,0,0,14,2


In [ ]:
X = df_selected[['value']]
y = df_selected['risk']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    max_depth=None
)

model.fit(X_train, y_train)


RandomForestClassifier(n_estimators=300, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9953488372093023
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        49
           1       1.00      0.98      0.99        43
           2       0.99      1.00      0.99        71
           3       1.00      1.00      1.00         1
           4       1.00      1.00      1.00        51

    accuracy                           1.00       215
   macro avg       1.00      1.00      1.00       215
weighted avg       1.00      1.00      1.00       215



In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

from google.colab import files
files.download("model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>